# Chadstone Parking Occupancy Analysis

This notebook analyses the historical occupancy of the Chadstone Shopping Centre car parks.
The data is scraped on a regular schedule and published to [`data/parking.csv`](../data/parking.csv)
in this repository.

Each row records how many spaces are **occupied** and **vacant** in a given car park at a point in time.
The `total_occupied` and `total_vacant` columns show the site-wide totals for that timestamp.

We use [Polars](https://pola.rs) for data handling and the lightweight [`xy`](https://github.com/factorish/xy) library for charts.

## 1. Setup

Import the libraries used throughout the notebook.

In [ ]:
import polars as pl
import xy

## 2. Load the data

Read the latest parking history directly from the GitHub-hosted CSV, then parse
`retrieved_at` into a proper datetime column.

In [ ]:
df = pl.read_csv(
    "https://github.com/jay-stein/chaddy-parking-scraper/blob/master/data/parking.csv?raw=true"
).with_columns(
    pl.col("retrieved_at").str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S")
)

df

## 3. Explore the data

Get a feel for the shape of the dataset: the number of rows, the available car parks,
and the time range covered.

In [ ]:
df.shape

In [ ]:
df.select("car_park").unique().sort("car_park")

In [ ]:
df.select(pl.col("retrieved_at").min().alias("earliest"), pl.col("retrieved_at").max().alias("latest"))

## 4. Car Park B occupancy over time

Filter down to Car Park B and sort by time so we can chart its occupancy trend.

In [ ]:
subset_df = df.filter(pl.col("car_park") == "B").sort("retrieved_at")

subset_df

## 5. Plot Car Park B

Draw a line chart of occupied spaces in Car Park B over time using `xy`.

In [ ]:
chart = xy.line_chart(
    xy.line(
        subset_df["retrieved_at"],
        subset_df["occupied"],
        color="#7c3aed",
        width=3,
    ),
    xy.x_axis(label="Time"),
    xy.y_axis(label="Occupied Spaces"),
    title="Occupied Spaces in Car Park B Over Time",
)

chart

## 6. Export the chart to HTML

Save the chart as a self-contained HTML file so it can be viewed outside the notebook.

In [ ]:
from pathlib import Path

Path("../charts/carpark_b.html").write_text(chart.to_html(), encoding="utf-8")

## 7. Total occupancy across all car parks

Sum the occupied spaces over every car park per timestamp to see how busy the
whole centre is at each sample.

In [ ]:
total_df = (
    df.group_by("retrieved_at")
    .agg(pl.sum("occupied").alias("total_occupied"))
    .sort("retrieved_at")
)

total_df

In [ ]:
total_chart = xy.line_chart(
    xy.line(
        total_df["retrieved_at"],
        total_df["total_occupied"],
        color="#0ea5e9",
        width=3,
    ),
    xy.x_axis(label="Time"),
    xy.y_axis(label="Total Occupied Spaces"),
    title="Total Occupancy Across All Car Parks",
)

total_chart